In [ ]:
import numpy as np
import sys
sys.path.extend([
    "/mnt/pve/Homes/bapun/Codes/BanditPy",
    "/mnt/pve/Homes/bapun/Codes/NeuroPy",
    "/mnt/pve/Homes/bapun/Codes/py_adlab_bg"
])

# from banditpy.models.rnn_models import TinyBehaviorRNN, TinyBehaviorRNNTrainer
from banditpy.models.rnn_models import TinyBehaviorRNN, TinyBehaviorRNNTrainer


# Build some synthetic sessions
def gen_session(T=200, p=(0.3, 0.7), seed=None):
    rng = np.random.default_rng(seed)
    actions = []
    rewards = []
    last = rng.integers(0, 2)
    for t in range(T):
        # simple biased policy
        if rng.random() < 0.1:  # exploration
            a = rng.integers(0, 2)
        else:
            a = last
        r = 1 if rng.random() < p[a] else 0
        actions.append(a)
        rewards.append(r)
        last = a if r == 1 else last
    return {"actions": np.array(actions), "rewards": np.array(rewards)}


train_sessions = [gen_session(seed=i) for i in range(20)]
val_sessions = [gen_session(seed=100 + i) for i in range(5)]
test_sessions = [gen_session(seed=200 + i) for i in range(5)]

model = TinyBehaviorRNN(
    input_size=3, num_actions=2, hidden_size=2, diagonal_readout=False
)
trainer = TinyBehaviorRNNTrainer(
    model, lr=5e-3, weight_decay=5e-4, patience=15, max_epochs=300
)

history = trainer.fit(train_sessions, val_sessions)
metrics = trainer.evaluate(test_sessions)
print("Final test NLL:", metrics["test_nll"])
print("Train NLL history (last 5):", history["train_nll"][-5:])
print("Val NLL history (last 5):", history["val_nll"][-5:])

In [1]:
import sys
sys.path.extend([
    "/mnt/pve/Homes/bapun/Codes/BanditPy",
    "/mnt/pve/Homes/bapun/Codes/NeuroPy",
    "/mnt/pve/Homes/bapun/Codes/py_adlab_bg"
])
from banditpy.core.mab import Bandit2Arm
from banditpy.models.rnn_models import nested_cross_validation_tiny_behavior_v2, tiny_behavior_d_vs_weighted_nll
import numpy as np
import time
import mab_subjects

#############################################
# Quick synthetic sanity check (3 sessions) #
#############################################
def make_synthetic_sessions(n_sessions=3, trials_per=240, p=(0.3,0.7), seed=0):
    rng = np.random.default_rng(seed)
    sessions = []
    for s in range(n_sessions):
        actions = []
        rewards = []
        last = rng.integers(0,2)
        for t in range(trials_per):
            if rng.random() < 0.15:
                a = rng.integers(0,2)
            else:
                a = last
            r = 1 if rng.random() < p[a] else 0
            actions.append(a)
            rewards.append(r)
            # simple win-stay tendency
            if r == 1:
                last = a
        sessions.append({'actions': np.array(actions), 'rewards': np.array(rewards)})
    return sessions

synthetic_sessions = make_synthetic_sessions()
print('Synthetic sessions lens:', [len(s['actions']) for s in synthetic_sessions])

# Run nested CV v2 on synthetic (small grids for speed)
start = time.time()
res = nested_cross_validation_tiny_behavior_v2(
    data=synthetic_sessions,
    num_actions=2,
    hidden_size_grid=(1,2),
    l1_grid=(1e-5,1e-4),
    seed_grid=(0,1),
    block_size=120,
    outer_folds=5,
    patience=50,
    max_epochs=1500,
    weight_inner_by_trials=True,
    selection_metric='val',
    derive_refit_epoch='median',
    checkpoint_path=None,
    status_every=1,
    verbose=True,
    include_zero_l1=True,
 )
elapsed = time.time() - start
print(f'Nested CV synthetic run took {elapsed:.2f}s')
print('d vs weighted test NLL:', tiny_behavior_d_vs_weighted_nll(res))
for d, info in res['per_d'].items():
    print('d', d, 'weighted mean test NLL', info['weighted_mean_test_nll'])

#############################################
# Existing real data example (commented)    #
#############################################
# Suppose you already constructed a Bandit2Arm object named task
# sess = mab_subjects.unstruc.allsess[0]
# cv_result = nested_cross_validation_tiny_behavior_v2(
#     data=sess.b2a.filter_by_trials(100,100),
#     num_actions=2,
#     hidden_size_grid=(2,4),
#     l1_grid=(1e-5,1e-4,1e-3),
#     seed_grid=(0,1,2),
#     block_size=100,
#     outer_folds=10,
#     patience=200,
#     max_epochs=5000,
#     include_zero_l1=False,
#     weight_inner_by_trials=True,
#     derive_refit_epoch='median',
#     checkpoint_path='subject1_nestedcv.json',
#     resume=False,
# )
# print('Subject weighted mean per d:', tiny_behavior_d_vs_weighted_nll(cv_result))

Synthetic sessions lens: [240, 240, 240]
[Fold 0 | d=1] test NLL=0.2176 l1=0.0 seed=0 refit_epochs=1499
[Fold 0 | d=1] test NLL=0.2176 l1=0.0 seed=0 refit_epochs=1499
[Fold 0 | d=2] test NLL=0.2129 l1=0.0001 seed=1 refit_epochs=706
[Fold 0 | d=2] test NLL=0.2129 l1=0.0001 seed=1 refit_epochs=706
[Fold 1 | d=1] test NLL=0.2060 l1=0.0 seed=0 refit_epochs=1149
[Fold 1 | d=1] test NLL=0.2060 l1=0.0 seed=0 refit_epochs=1149
[Fold 1 | d=2] test NLL=0.2283 l1=0.0001 seed=0 refit_epochs=1415
[Fold 1 | d=2] test NLL=0.2283 l1=0.0001 seed=0 refit_epochs=1415
[Fold 2 | d=1] test NLL=0.2382 l1=0.0 seed=0 refit_epochs=1175
[Fold 2 | d=1] test NLL=0.2382 l1=0.0 seed=0 refit_epochs=1175
[Fold 2 | d=2] test NLL=0.2340 l1=0.0 seed=1 refit_epochs=874
[Fold 2 | d=2] test NLL=0.2340 l1=0.0 seed=1 refit_epochs=874
[Fold 3 | d=1] test NLL=0.2582 l1=0.0 seed=0 refit_epochs=1176
[Fold 3 | d=1] test NLL=0.2582 l1=0.0 seed=0 refit_epochs=1176
[Fold 3 | d=2] test NLL=0.2517 l1=0.0 seed=0 refit_epochs=1176
[Fold 

In [6]:

a = tiny_behavior_d_vs_weighted_nll(res)

In [8]:
import pandas as pd

pd.DataFrame(a,columns=['d','nll'])

,d,nll
0,1,0.232048
1,2,0.233372


In [3]:
# Analyze lag-1 autocorrelation from *_tier1.json files and compare groups
import glob, json, os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# Configuration
TIER1_GLOB = os.path.join("/mnt/pve/Homes/bapun/Codes/py_adlab_server/slurm_scripts/", "*_tier1.json")

paths = sorted(glob.glob(TIER1_GLOB))
if not paths:
    print("No *_tier1.json files found in", os.getcwd())
else:
    rows = []
    for p in paths:
        if not p.endswith('_tier1.json'):  # safety
            continue
        try:
            with open(p, 'r') as f:
                data = json.load(f)
        except Exception as e:
            print(f"[WARN] Failed to read {p}: {e}")
            continue
        subject = data.get('subject', os.path.basename(p).replace('_tier1.json',''))
        group = data.get('group', 'unknown')
        first_exp = data.get('first_experience', False)
        tier1_summary = data.get('tier1_summary', {})
        for d_str, info in tier1_summary.items():
            lag_vec = info.get('lag1_autocorr')
            if lag_vec is None:
                continue
            try:
                lag_arr = np.array(lag_vec, dtype=float)
            except Exception:
                continue
            if lag_arr.size == 0:
                continue
            rows.append({
                'subject': subject,
                'group': group,
                'first_experience': first_exp,
                'd': int(d_str),
                'lag1_mean': float(lag_arr.mean()),
                'lag1_median': float(np.median(lag_arr)),
                'lag1_abs_mean': float(np.mean(np.abs(lag_arr))),
                'lag1_std': float(lag_arr.std()),
                'n_units': int(lag_arr.size),
            })
    if not rows:
        print("No lag1_autocorr data extracted.")
    else:
        df = pd.DataFrame(rows)
        display(df.head())
        # Subject-level aggregation (already mean per subject, but keep as is)
        # Group-level statistics
        grp = df.groupby(['group','d']).agg(
            subjects=('subject','nunique'),
            mean_lag1=('lag1_mean','mean'),
            sem_lag1=('lag1_mean', lambda x: x.std(ddof=1)/np.sqrt(len(x)) if len(x)>1 else np.nan),
            mean_abs_lag1=('lag1_abs_mean','mean'),
        ).reset_index()
        display(grp)

        # Plot mean lag1 with error bars (SEM)
        plt.figure(figsize=(4,3.2))
        colors = {'struc':'#16a085','unstruc':'#e67e22','unknown':'#7f8c8d'}
        for g, gdf in grp.groupby('group'):
            gdf = gdf.sort_values('d')
            plt.errorbar(gdf['d'], gdf['mean_lag1'], yerr=gdf['sem_lag1'], marker='o', capsize=3,
                         label=g, color=colors.get(g,'#555555'))
        plt.xlabel('d')
        plt.ylabel('mean lag1 autocorr (per-subject mean)')
        plt.title('Hidden state lag-1 autocorrelation by group')
        plt.legend(frameon=False)
        plt.tight_layout()
        plt.show()

        # Optional: simple difference table (struc - unstruc) if both present
        if set(grp['group']) >= {'struc','unstruc'}:
            pivot = grp.pivot(index='d', columns='group', values='mean_lag1')
            pivot['diff_struc_minus_unstruc'] = pivot['struc'] - pivot['unstruc']
            print("Difference (struc - unstruc):")
            display(pivot[['diff_struc_minus_unstruc']])

plt.show()

,subject,group,first_experience,d,lag1_mean,lag1_median,lag1_abs_mean,lag1_std,n_units
0,AggroExp1Unstructured,unstruc,True,1,0.978682,0.978682,0.978682,0.0,1
1,AuromaExp1Unstructured,unstruc,True,1,0.965279,0.965279,0.965279,0.0,1
2,BewilderbeastExp1Structured,struc,True,1,0.933323,0.933323,0.933323,0.0,1
3,BratExp1Unstructured,unstruc,True,1,0.961717,0.961717,0.961717,0.0,1
4,BuffalordExp1Structured,struc,True,1,0.975944,0.975944,0.975944,0.0,1


,group,d,subjects,mean_lag1,sem_lag1,mean_abs_lag1
0,struc,1,5,0.960379,0.007654,0.960379
1,unstruc,1,6,0.965666,0.004015,0.965666


Difference (struc - unstruc):


group,diff_struc_minus_unstruc
d,
1,-0.005287


In [8]:
df

,subject,group,first_experience,d,lag1_mean,lag1_median,lag1_abs_mean,lag1_std,n_units
0,AggroExp1Unstructured,unstruc,True,1,0.978682,0.978682,0.978682,0.0,1
1,AuromaExp1Unstructured,unstruc,True,1,0.965279,0.965279,0.965279,0.0,1
2,BewilderbeastExp1Structured,struc,True,1,0.933323,0.933323,0.933323,0.0,1
3,BratExp1Unstructured,unstruc,True,1,0.961717,0.961717,0.961717,0.0,1
4,BuffalordExp1Structured,struc,True,1,0.975944,0.975944,0.975944,0.0,1
5,GronckleExp1Structured,struc,True,1,0.970296,0.970296,0.970296,0.0,1
6,GronckleExp2Unstructured,unstruc,False,1,0.974371,0.974371,0.974371,0.0,1
7,GrumpExp1Unstructured,unstruc,True,1,0.962949,0.962949,0.962949,0.0,1
8,GrumpExp2Structured,struc,False,1,0.968157,0.968157,0.968157,0.0,1
9,ToothlessExp1Structured,struc,True,1,0.954176,0.954176,0.954176,0.0,1
